# Kaggriculture Mechanics — the Numbers Behind the Game

A short, runnable reference for the numbers the environment doesn't spell out on the
competition page. No strategies here; just the mechanics you can compute directly from
the environment source, so you can decide for yourself.

Three tables:

1. **Product price curves** — what each product actually fetches as inventory changes.
2. **Crop value per tile-day** — throughput at base price.
3. **Animal yield with `CARE`** — including the compounding rule that flips the ranking.

---
## 1. Product price curves

`market_price(item, inventory)` is the game's own pricing function. Prices move as inventory
does — buying and town consumption raise them; your sales lower them. This is how the price
looks after you (net) sell N units into the market above the starting inventory.

In [ ]:
from kaggle_environments.envs.kaggriculture.kaggriculture import (
    market_price, MARKET_I0 as I0, CROPS, ANIMALS,
)

ITEMS = ["WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON", "EGG", "MILK", "WOOL", "FERTILIZER"]
VOLUMES = [0, 25, 50, 75, 100, 150, 200, 300]

print(f"{'item':<12}" + "".join(f"{v:>7}" for v in VOLUMES))
print(f"{'':12}" + "".join(f"{'sold':>7}" for _ in VOLUMES))
print("-" * (12 + 7 * len(VOLUMES)))
for item in ITEMS:
    row = "".join(f"{market_price(item, I0 + v):>7}" for v in VOLUMES)
    print(f"{item:<12}{row}")

Two curve shapes are visible:

* **Slow-decay products** (`WHEAT`, `EGG`, `FERTILIZER`) hold most of their price out past 200
  units.
* **Fast-decay products** (`STRAWBERRY`, `MELON`, `MILK`, `WOOL`) reach the $1 floor somewhere
  in the 75–150 range.

The exact shape per product lives in `MARKET_PARAMS` in the environment source.

---
## 2. Crop value per tile-day

Base-price throughput per tile, ignoring price movement (from the previous table) and
watering/labour cost. `max_yield` assumes fertilizer — unfertilized, wheat peaks at 4 and
carrot at 3, so those two rows are optimistic.

In [ ]:
print(f"{'crop':<12}{'seed':>6}{'units':>7}{'days':>6}{'$/tile/day':>12}")
print("-" * 43)
for name, cd in CROPS.items():
    base = market_price(name, I0)
    days = max(cd["max_yield_day"], 1)
    units = cd["max_yield"]
    print(f"{name:<12}{cd['seed']:>6}{units:>7}{days:>6}{units * base / days:>12.0f}")

Base-price throughput and price stability point in different directions — this table only
tells you the first.

---
## 3. Animal yield with `CARE`

The compounding rule from the environment README:

> `CARE` banks a yield bonus that is paid out on the animal's next scheduled production.
> At end of day, if the animal was both fed AND cared for that day, `pending_care_bonus`
> increments by 1. On a scheduled production day, if the animal is fed, the entire banked
> bonus is added to that production's yield.

So the slower an animal produces, the more `CARE` bonus has accrued by payout. With daily
`CARE`, per-yield output is `1 + interval` (base 1 plus one banked point per intervening day).

In [ ]:
print(f"{'animal':<8}{'cost':>6}{'product':>9}{'base':>6}{'every':>7}{'per yield':>11}{'$/day':>8}")
print("-" * 55)
for name, a in ANIMALS.items():
    base = market_price(a["product"], I0)
    iv = a["interval"]
    per_yield = 1 + iv          # base 1, plus one banked CARE point per intervening day
    per_day = per_yield / iv
    print(f"{name:<8}{a['cost']:>6}{a['product']:>9}{base:>6}{iv:>7}{per_yield:>11}{per_day * base:>8.0f}")

Every surviving animal also drops **1 fertilizer per day**, whether or not it was fed or
cared for. The fertilizer price is in the first table.

Cost recovery at base price, without `CARE`, is roughly:

* `GOOSE`  — $300 / 50 = 6 production days (daily)
* `COW`    — $400 / 160 = 3 production days = 6 real days (every 2)
* `SHEEP`  — $500 / 200 = 3 production days = 9 real days (every 3)

The `interval` column is the missing piece of that arithmetic on the competition page.

---
That's the mechanics. Sources are `kaggle_environments/envs/kaggriculture/kaggriculture.py`
(functions `market_price`, and constants `CROPS`, `ANIMALS`, `MARKET_PARAMS`) and the
package `README.md` next to it. Corrections welcome.